# 03 — Evaluation & Head B/C Analysis

**Boréal Marché — Evaluation Across All Three Heads**

This notebook consolidates evaluation results for all three model heads:
- Head A: Retention prediction metrics and error analysis
- Head B: Association rules — support sweep, hit-rate@5, coverage
- Head C: Assistant evaluation — tool-choice accuracy by provider

All business logic lives in `src/` — this notebook calls those modules
and visualises the output.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
ROOT = Path("..").resolve()

## Head B — Association Rules

### Support Sweep Results

In [ ]:
# Load the rules artifact
rules_path = ROOT / "artifacts" / "rules.json"
if rules_path.exists():
    with open(rules_path) as f:
        rules_data = json.load(f)
    header = rules_data["header"]
    print(f"Mining date: {header['mining_date']}")
    print(f"Algorithm: {header['algorithm_prod']}")
    print(f"Rule count: {header['rule_count']}")
    print(f"\nMemory report:")
    for k, v in header["memory_report"].items():
        print(f"  {k}: {v}")
    print(f"\nTiming:")
    print(f"  Apriori:  {header['timing']['apriori_seconds']:.2f}s")
    print(f"  FP-Growth: {header['timing']['fpgrowth_seconds']:.2f}s")
else:
    print("No rules artifact found — run `make rules` first.")

### Discarded Rule Example (High-Confidence, Low-Lift)

In [ ]:
if rules_path.exists() and header.get("discarded_rule_examples"):
    for ex in header["discarded_rule_examples"]:
        print(f"Antecedent: {ex['antecedent']}")
        print(f"Consequent: {ex['consequent']}")
        print(f"Confidence: {ex['confidence']:.2%}")
        print(f"Lift:       {ex['lift']:.4f}")
        print(f"Reason:     {ex['reason_discarded']}")
else:
    print("No discarded rule examples available.")

### Hit-Rate@5 and Coverage

Evaluate on a temporal holdout: the last month of transactions.

In [ ]:
from src.recommend import RecommendationEngine

if rules_path.exists():
    engine = RecommendationEngine.load()
    print(f"Loaded {engine.n_rules} rules, {engine.n_indexed_items} indexed items")

    # Load transactions for holdout evaluation
    df = pd.read_parquet(ROOT / "data/processed/transactions.parquet")
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

    # Use last 30 days as holdout
    max_date = df["InvoiceDate"].max()
    holdout = df[df["InvoiceDate"] >= max_date - pd.Timedelta(days=30)]

    # Group by invoice to create baskets
    holdout_baskets = (
        holdout.groupby("Invoice")["Description"]
        .apply(list)
        .tolist()
    )

    # For each basket with >= 2 items, hold out one item and check if it's recommended
    hits = 0
    total = 0
    strategies = {"rule_based": 0, "popularity_fallback": 0}

    for basket in holdout_baskets:
        if len(basket) < 2:
            continue
        held_out = basket[-1]
        query = basket[:-1]
        result = engine.recommend(query, top_k=5)
        strategies[result["strategy"]] = strategies.get(result["strategy"], 0) + 1
        if held_out.lower().strip() in [r.lower().strip() for r in result["recommendations"]]:
            hits += 1
        total += 1

    hit_rate = hits / total if total > 0 else 0
    coverage = strategies.get("rule_based", 0) / total if total > 0 else 0
    print(f"\nHit-rate@5: {hit_rate:.4f} ({hits}/{total})")
    print(f"Coverage (rule-based): {coverage:.2%}")
    print(f"Strategies: {strategies}")
else:
    print("No rules artifact — run `make rules` first.")

## Head C — Assistant Evaluation

### Golden Set Tool-Choice Accuracy

In [ ]:
import asyncio
from src.rag.providers import RegexProvider, get_provider
from src.rag.guardrails import check_out_of_scope, detect_language

golden_path = ROOT / "tests" / "golden" / "golden_questions.json"
with open(golden_path) as f:
    golden = json.load(f)

questions = golden["questions"]
print(f"Golden set: {len(questions)} questions")
print(f"Breakdown: {golden['breakdown']}")

In [ ]:
async def evaluate_provider(provider, questions):
    results = []
    for q in questions:
        if q["category"] == "out_of_scope":
            guardrail = check_out_of_scope(q["text"])
            results.append({
                "id": q["id"],
                "category": q["category"],
                "expected": None,
                "predicted": None if not guardrail.passed else "not_refused",
                "correct": not guardrail.passed,
            })
        else:
            result = await provider.answer(q["text"], [], [], q["language"])
            results.append({
                "id": q["id"],
                "category": q["category"],
                "expected": q["expected_tool"],
                "predicted": result.tool_name,
                "correct": result.tool_name == q["expected_tool"],
            })
    return results

# Evaluate regex provider
regex = RegexProvider()
regex_results = asyncio.get_event_loop().run_until_complete(
    evaluate_provider(regex, questions)
)

regex_df = pd.DataFrame(regex_results)
print("\n=== Regex Provider ===")
for cat in ["policy", "tool_call", "out_of_scope", "french"]:
    subset = regex_df[regex_df["category"] == cat]
    acc = subset["correct"].mean()
    print(f"  {cat}: {acc:.0%} ({subset['correct'].sum()}/{len(subset)})")
print(f"  Overall: {regex_df['correct'].mean():.0%}")

### Language Detection Accuracy

In [ ]:
lang_results = []
for q in questions:
    detected = detect_language(q["text"])
    lang_results.append({
        "id": q["id"],
        "expected": q["language"],
        "detected": detected,
        "correct": detected == q["language"],
    })

lang_df = pd.DataFrame(lang_results)
print(f"Language detection accuracy: {lang_df['correct'].mean():.0%}")
print(f"  English: {lang_df[lang_df['expected']=='en']['correct'].mean():.0%}")
print(f"  French:  {lang_df[lang_df['expected']=='fr']['correct'].mean():.0%}")

### Summary Table

In [ ]:
summary = pd.DataFrame({
    "Provider": ["Regex"],
    "Tool-choice accuracy": [f"{regex_df[regex_df['category'] != 'out_of_scope']['correct'].mean():.0%}"],
    "OOS refusal rate": [f"{regex_df[regex_df['category'] == 'out_of_scope']['correct'].mean():.0%}"],
    "Cost/1K queries": ["$0"],
})
# Add OpenAI and Ollama rows if available
print("\n=== Provider Comparison ===")
summary